In [ ]:
## we test to see if taylor hood can solve the source problem. 

In [2]:
from firedrake import *
import numpy as np

# ==============================================================================
# PART 1: MMS Convergence Test (Proving LBB Stability for the Source Problem)
# ==============================================================================
def run_mms_convergence():
    print("--- Running MMS Convergence for Taylor-Hood (CG2 x CG1) ---")
    L_domain = np.pi
    h_vals = []
    error_v_vals = []
    error_p_vals = []
    
    # Loop over mesh refinements
    for N in [4, 8, 16, 32]:
        mesh = SquareMesh(N, N, L_domain, quadrilateral=False, diagonal='crossed')
        h_vals.append(L_domain / N)
        
        # Taylor-Hood Spaces: CG2 for the vector field, CG1 for the scalar multiplier
        V = VectorFunctionSpace(mesh, "CG", 2)
        Q = FunctionSpace(mesh, "CG", 1)
        W = V * Q
        
        x, y = SpatialCoordinate(mesh)
        
        # 1. Manufacture Exact Solutions 
        # These vanish on the boundaries of [0, pi]^2 to satisfy Dirichlet BCs
        v_exact = as_vector([sin(x)*sin(y), sin(x)*sin(y)])
        p_exact = sin(x)*sin(y)
        
        eps = Constant(1.0)
        
        # 2. Derive the exact right-hand side source terms
        # f = curl(curl(v)) + grad(p)
        f_exact = eps * curl(curl(v_exact)) + grad(p_exact)
        
        # 3. Setup the Variational Problem
        v, p = TrialFunctions(W)
        w, q = TestFunctions(W)
        
        # Left-hand side: The B-formulation saddle point system
        a = eps * inner(curl(v), curl(w))*dx + inner(grad(p), w)*dx + inner(v, grad(q))*dx
        
        # Right-hand side: We use the exact expressions directly against the test functions
        L = inner(f_exact, w)*dx + inner(v_exact, grad(q))*dx
        
        sol = Function(W)
        
        # 4. Apply Strong Dirichlet BCs to all continuous components
        bcs = [
            DirichletBC(W.sub(0).sub(1), 0, [1, 2]), 
            DirichletBC(W.sub(0).sub(0), 0, [3, 4]),
            DirichletBC(W.sub(1), 0, "on_boundary")
        ]
        
        # Solve the linear system
        solve(a == L, sol, bcs=bcs, solver_parameters={'ksp_type': 'preonly', 'pc_type': 'lu'})
        
        v_sol, p_sol = sol.subfunctions
        
        # 5. Compute L2 Errors
        err_v = errornorm(v_exact, v_sol, norm_type="L2")
        err_p = errornorm(p_exact, p_sol, norm_type="L2")
        
        error_v_vals.append(err_v)
        error_p_vals.append(err_p)
        
        print(f"N={N:2d}: L2 Error v = {err_v:.4e}, L2 Error p = {err_p:.4e}")
        
    # 6. Calculate and print Convergence Rates (should be ~3.0 for v, ~2.0 for p)
    print("\n--- Convergence Rates ---")
    for i in range(1, len(h_vals)):
        rate_v = np.log2(error_v_vals[i-1] / error_v_vals[i])
        rate_p = np.log2(error_p_vals[i-1] / error_p_vals[i])
        print(f"N={4*(2**i):2d}: Rate v = {rate_v:.2f}, Rate p = {rate_p:.2f}")

# Execute the MMS test
run_mms_convergence()


# ==============================================================================
# PART 2: Eigenvalue Trap (Proving DCP Failure for Spectral Problem)
# ==============================================================================
# You can plug this directly into your existing solver script.

def taylor_hood_elements(mesh, degree):
    # Pass degree=1 to create CG2 x CG1
    return VectorFunctionSpace(mesh, "CG", degree + 1) * FunctionSpace(mesh, "CG", degree)

def taylor_hood_bc(W):
    # Because both are CG, we constrain the multiplier sub(1) as well
    return [
        DirichletBC(W.sub(0).sub(1), 0, [1, 2]), 
        DirichletBC(W.sub(0).sub(0), 0, [3, 4]),
        DirichletBC(W.sub(1), 0, "on_boundary")
    ]

--- Running MMS Convergence for Taylor-Hood (CG2 x CG1) ---


ConvergenceError: Nonlinear solve failed to converge after 0 nonlinear iterations.
Reason:
   DIVERGED_LINEAR_SOLVE